[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/ONNX_Tutorial/blob/main/03_ONNX_Architecture_and_Internals/04_Type_System_and_Shapes/Type_System_and_Shapes_Apply.ipynb)

# 3.4 Type System and Shapes — Hands-On Practice

## Objective

Master ONNX's **type system** (element types, tensor types) and **shape inference**.
Learn symbolic dimensions, dynamic shapes, and how to debug shape mismatches.

---

| # | Section | Focus |
|---|---------|-------|
| 1 | Setup | Dependencies |
| 2 | Exercise 1: Symbolic Batch Dimensions | Dynamic batch via `dim_param` |
| 3 | Exercise 2: Shape Inference | Propagate shapes automatically |
| 4 | Exercise 3: Inspect All Types | Walk TypeProto fields |
| 5 | Exercise 4: Mixed-Type Models | INT64 indices with FLOAT data |
| 6 | Exercise 5: Concrete vs Symbolic | Fixed shapes vs flexible |
| 7 | Exercise 6: Shape Debugging | Catch dimension mismatches |
| 8 | Exercise 7: Dynamic Axis Testing | Run with varying sizes |
| 9 | Challenge: Shape Analyzer Tool | Complete model shape audit |

In [ ]:
# !pip install onnx onnxruntime numpy matplotlib

import numpy as np
import time
import onnx
from onnx import TensorProto, shape_inference
from onnx.helper import (
    make_model, make_node, make_graph,
    make_tensor_value_info, make_opsetid)
from onnx.checker import check_model
from onnx.numpy_helper import from_array, to_array
import onnxruntime as ort

print(f'ONNX: {onnx.__version__}  ORT: {ort.__version__}')

## Exercise 1: Symbolic Batch Dimensions

In ONNX, dimensions can be:
- **Fixed** (integer): `[1, 3, 224, 224]` — concrete shapes
- **Symbolic** (string): `['N', 3, 'H', 'W']` — dynamic, resolved at runtime

Symbolic dimensions let one `.onnx` file handle any batch size.

In [ ]:
# Model with symbolic batch dimension 'N'
W = from_array(np.random.randn(4, 3).astype(np.float32), 'W')

X = make_tensor_value_info('X', TensorProto.FLOAT, ['N', 4])
Y = make_tensor_value_info('Y', TensorProto.FLOAT, ['N', 3])

g = make_graph(
    [make_node('MatMul', ['X', 'W'], ['Y'])],
    'dynamic_batch', [X], [Y], [W])
model_dyn = make_model(g, opset_imports=[make_opsetid('', 17)])
check_model(model_dyn)

# Inspect type information
def print_type_info(vi, label=''):
    t = vi.type.tensor_type
    dtype = TensorProto.DataType.Name(t.elem_type)
    if t.HasField('shape'):
        dims = []
        for d in t.shape.dim:
            if d.dim_param:
                dims.append(f'{d.dim_param} (symbolic)')
            elif d.dim_value:
                dims.append(f'{d.dim_value} (fixed)')
            else:
                dims.append('? (unknown)')
        print(f'{label}{vi.name}: {dtype} [{" × ".join(dims)}]')
    else:
        print(f'{label}{vi.name}: {dtype} [shape unknown]')

print('Input type info:')
for inp in model_dyn.graph.input:
    print_type_info(inp, '  ')
print('Output type info:')
for out in model_dyn.graph.output:
    print_type_info(out, '  ')

# Test with different batch sizes
sess = ort.InferenceSession(
    model_dyn.SerializeToString(), providers=['CPUExecutionProvider'])

for bs in [1, 5, 20, 100]:
    x = np.random.randn(bs, 4).astype(np.float32)
    y = sess.run(None, {'X': x})[0]
    assert y.shape == (bs, 3)
    print(f'  batch={bs:4d}: input={x.shape} → output={y.shape}')

print('All batch sizes work with symbolic dim!')

## Exercise 2: Shape Inference

Shape inference propagates known shapes through the graph, filling in
intermediate `value_info` entries.

In [ ]:
np.random.seed(42)

W1 = from_array(np.random.randn(4, 8).astype(np.float32), 'W1')
W2 = from_array(np.random.randn(8, 3).astype(np.float32), 'W2')

X = make_tensor_value_info('X', TensorProto.FLOAT, ['N', 4])
Y = make_tensor_value_info('Y', TensorProto.FLOAT, None)  # unknown!

g = make_graph([
    make_node('MatMul', ['X', 'W1'], ['H']),
    make_node('Relu', ['H'], ['H_act']),
    make_node('MatMul', ['H_act', 'W2'], ['Y']),
], 'infer_demo', [X], [Y], [W1, W2])

model_pre = make_model(g, opset_imports=[make_opsetid('', 17)])
check_model(model_pre)

print('BEFORE shape inference:')
print(f'  value_info: {len(model_pre.graph.value_info)} entries')
print(f'  Output shape known: {model_pre.graph.output[0].type.tensor_type.HasField("shape")}')

# Run shape inference
model_post = shape_inference.infer_shapes(model_pre)

print(f'\nAFTER shape inference:')
print(f'  value_info: {len(model_post.graph.value_info)} entries')

for vi in model_post.graph.value_info:
    print_type_info(vi, '    ')

print('  Output:')
print_type_info(model_post.graph.output[0], '    ')

## Exercise 3: Inspect All Types

Walk through every tensor in the model and display full type information.

In [ ]:
def full_type_audit(model):
    """Print complete type information for every tensor in the model."""
    shaped = shape_inference.infer_shapes(model)
    init_names = {i.name for i in shaped.graph.initializer}

    print(f'{"Tensor":15s} {"Category":12s} {"Type":8s} {"Shape"}')
    print('=' * 55)

    # Inputs
    for inp in shaped.graph.input:
        if inp.name in init_names:
            continue
        t = inp.type.tensor_type
        dtype = TensorProto.DataType.Name(t.elem_type)
        dims = [d.dim_param or d.dim_value for d in t.shape.dim] if t.HasField('shape') else '?'
        print(f'  {inp.name:15s} {"input":12s} {dtype:8s} {dims}')

    # Initializers
    for init in shaped.graph.initializer:
        arr = to_array(init)
        print(f'  {init.name:15s} {"initializer":12s} '
              f'{str(arr.dtype):8s} {list(arr.shape)}')

    # Intermediates
    for vi in shaped.graph.value_info:
        t = vi.type.tensor_type
        dtype = TensorProto.DataType.Name(t.elem_type)
        dims = [d.dim_param or d.dim_value for d in t.shape.dim] if t.HasField('shape') else '?'
        print(f'  {vi.name:15s} {"intermediate":12s} {dtype:8s} {dims}')

    # Outputs
    for out in shaped.graph.output:
        t = out.type.tensor_type
        dtype = TensorProto.DataType.Name(t.elem_type)
        dims = [d.dim_param or d.dim_value for d in t.shape.dim] if t.HasField('shape') else '?'
        print(f'  {out.name:15s} {"output":12s} {dtype:8s} {dims}')

full_type_audit(model_post)

## Exercise 4: Mixed-Type Models

Real models mix types: FLOAT for data, INT64 for indices, BOOL for masks.

In [ ]:
# Gather: selects rows from a matrix using INT64 indices
# data[indices] where data is FLOAT, indices is INT64

data_input = make_tensor_value_info('data', TensorProto.FLOAT, [10, 4])
idx_input = make_tensor_value_info('indices', TensorProto.INT64, [3])
out = make_tensor_value_info('output', TensorProto.FLOAT, [3, 4])

g = make_graph(
    [make_node('Gather', ['data', 'indices'], ['output'], axis=0)],
    'mixed_types', [data_input, idx_input], [out])
m_mixed = make_model(g, opset_imports=[make_opsetid('', 17)])
check_model(m_mixed)

sess_m = ort.InferenceSession(
    m_mixed.SerializeToString(), providers=['CPUExecutionProvider'])

data = np.arange(40, dtype=np.float32).reshape(10, 4)
indices = np.array([2, 5, 8], dtype=np.int64)
result = sess_m.run(None, {'data': data, 'indices': indices})[0]

print(f'data type:    {data.dtype}')
print(f'indices type: {indices.dtype}')
print(f'output type:  {result.dtype}')
print(f'\nGather rows {indices.tolist()} from 10×4 matrix:')
print(result)
assert np.allclose(result, data[indices])

# Where: uses BOOL condition with FLOAT data
print('\n--- Where (BOOL + FLOAT) ---')
C = make_tensor_value_info('C', TensorProto.BOOL, ['N'])
A = make_tensor_value_info('A', TensorProto.FLOAT, ['N'])
B = make_tensor_value_info('B', TensorProto.FLOAT, ['N'])
O = make_tensor_value_info('O', TensorProto.FLOAT, ['N'])

g_w = make_graph(
    [make_node('Where', ['C', 'A', 'B'], ['O'])],
    'where_test', [C, A, B], [O])
m_w = make_model(g_w, opset_imports=[make_opsetid('', 17)])

sess_w = ort.InferenceSession(
    m_w.SerializeToString(), providers=['CPUExecutionProvider'])
c = np.array([True, False, True, False], dtype=bool)
a = np.array([1, 2, 3, 4], dtype=np.float32)
b = np.array([-1, -2, -3, -4], dtype=np.float32)
r = sess_w.run(None, {'C': c, 'A': a, 'B': b})[0]
print(f'cond (bool):   {c}')
print(f'A (float32):   {a}')
print(f'B (float32):   {b}')
print(f'Where result:  {r}')
assert np.allclose(r, np.where(c, a, b))

## Exercise 5: Concrete vs Symbolic Shapes

Compare models with fully concrete shapes vs symbolic dimensions.

In [ ]:
W_init = from_array(np.random.randn(4, 3).astype(np.float32), 'W')

# Concrete: fixed batch=1
X_c = make_tensor_value_info('X', TensorProto.FLOAT, [1, 4])
Y_c = make_tensor_value_info('Y', TensorProto.FLOAT, [1, 3])
g_c = make_graph(
    [make_node('MatMul', ['X', 'W'], ['Y'])],
    'concrete', [X_c], [Y_c], [W_init])
m_concrete = make_model(g_c, opset_imports=[make_opsetid('', 17)])

# Symbolic: dynamic batch
X_s = make_tensor_value_info('X', TensorProto.FLOAT, ['N', 4])
Y_s = make_tensor_value_info('Y', TensorProto.FLOAT, ['N', 3])
g_s = make_graph(
    [make_node('MatMul', ['X', 'W'], ['Y'])],
    'symbolic', [X_s], [Y_s], [W_init])
m_symbolic = make_model(g_s, opset_imports=[make_opsetid('', 17)])

# Compare
print(f'{"":20s} {"Concrete":>12s} {"Symbolic":>12s}')
print('-' * 46)
print(f'  {"Input shape":20s} {"[1, 4]":>12s} {"[N, 4]":>12s}')
print(f'  {"Size (bytes)":20s} '
      f'{len(m_concrete.SerializeToString()):>12,} '
      f'{len(m_symbolic.SerializeToString()):>12,}')

# Test both
s_c = ort.InferenceSession(
    m_concrete.SerializeToString(), providers=['CPUExecutionProvider'])
s_s = ort.InferenceSession(
    m_symbolic.SerializeToString(), providers=['CPUExecutionProvider'])

print(f'\nBatch flexibility test:')
for bs in [1, 5, 20]:
    x = np.random.randn(bs, 4).astype(np.float32)
    # Symbolic always works
    r_s = s_s.run(None, {'X': x})[0]
    print(f'  batch={bs}: symbolic OK (shape={r_s.shape})', end='')
    # Concrete may fail for batch > 1
    try:
        r_c = s_c.run(None, {'X': x})[0]
        print(f', concrete OK')
    except Exception:
        print(f', concrete FAIL (fixed batch=1)')

## Exercise 6: Shape Debugging

Build a tool to detect common shape issues in ONNX models.

In [ ]:
def debug_shapes(model):
    """Detect shape-related issues in a model."""
    issues = []

    # 1. Check for unknown output shapes
    for out in model.graph.output:
        t = out.type.tensor_type
        if not t.HasField('shape'):
            issues.append(f'Output "{out.name}" has no shape info')

    # 2. Run shape inference and check intermediates
    try:
        shaped = shape_inference.infer_shapes(model)
        n_unknown = 0
        for node in shaped.graph.node:
            for out in node.output:
                found = False
                for vi in shaped.graph.value_info:
                    if vi.name == out:
                        found = True
                        break
                for o in shaped.graph.output:
                    if o.name == out:
                        found = True
                if not found:
                    n_unknown += 1
        if n_unknown > 0:
            issues.append(f'{n_unknown} intermediate shapes could not be inferred')
    except Exception as e:
        issues.append(f'Shape inference failed: {str(e)[:60]}')

    # 3. Check initializer shapes match declarations
    input_shapes = {}
    for inp in model.graph.input:
        t = inp.type.tensor_type
        if t.HasField('shape'):
            input_shapes[inp.name] = [
                d.dim_value for d in t.shape.dim if d.dim_value > 0]

    for init in model.graph.initializer:
        arr_shape = list(init.dims)
        if init.name in input_shapes:
            declared = input_shapes[init.name]
            if declared and declared != arr_shape:
                issues.append(
                    f'Initializer "{init.name}": data shape {arr_shape} '
                    f'!= declared {declared}')

    # Report
    if issues:
        print(f'Shape issues ({len(issues)}):')
        for i in issues:
            print(f'  ! {i}')
    else:
        print('No shape issues detected!')
    return len(issues) == 0

print('=== Good model ===')
debug_shapes(model_post)

print('\n=== Model with unknown output shape ===')
X = make_tensor_value_info('X', TensorProto.FLOAT, ['N', 4])
Y = make_tensor_value_info('Y', TensorProto.FLOAT, None)
g = make_graph([make_node('Relu', ['X'], ['Y'])], 'unk', [X], [Y])
m_unk = make_model(g, opset_imports=[make_opsetid('', 17)])
debug_shapes(m_unk)

## Exercise 7: Dynamic Axis Testing

In [ ]:
import matplotlib.pyplot as plt

# Build model with symbolic batch and sequence dims
W_rnn = from_array(np.random.randn(8, 4).astype(np.float32), 'W')

X = make_tensor_value_info('X', TensorProto.FLOAT, ['batch', 'seq_len', 8])
Y = make_tensor_value_info('Y', TensorProto.FLOAT, ['batch', 'seq_len', 4])

g = make_graph(
    [make_node('MatMul', ['X', 'W'], ['Y'])],
    'dynamic_test', [X], [Y], [W_rnn])
m_dyn = make_model(g, opset_imports=[make_opsetid('', 17)])
check_model(m_dyn)

sess_dyn = ort.InferenceSession(
    m_dyn.SerializeToString(), providers=['CPUExecutionProvider'])

# Test many shape combinations
configs = [(1, 10), (4, 10), (16, 10), (1, 50), (4, 50),
           (16, 50), (1, 200), (4, 200), (16, 200)]
latencies = []

print(f'{"Batch":>6s} {"SeqLen":>7s} {"Shape":>18s} {"Latency (us)":>12s}')
print('-' * 47)

for bs, seq in configs:
    x = np.random.randn(bs, seq, 8).astype(np.float32)
    sess_dyn.run(None, {'X': x})  # warmup
    times = []
    for _ in range(200):
        t0 = time.perf_counter()
        sess_dyn.run(None, {'X': x})
        times.append((time.perf_counter() - t0) * 1e6)
    avg = np.mean(times)
    latencies.append((bs * seq, avg))
    print(f'{bs:>6d} {seq:>7d} {str(x.shape):>18s} {avg:>12.1f}')

# Plot
elements, lats = zip(*latencies)
fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(elements, lats, s=80, c='#2E86C1', zorder=5)
ax.set_xlabel('Total Elements (batch × seq_len)', fontsize=12)
ax.set_ylabel('Latency (us)', fontsize=12)
ax.set_title('Dynamic Shape: Latency vs Input Size', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Challenge: Complete Shape Analyzer

Build a comprehensive shape analysis tool that audits all tensor shapes
in a model and generates a visual report.

In [ ]:
def shape_analyzer(model, run_test=True):
    """Comprehensive shape analysis tool."""
    shaped = shape_inference.infer_shapes(model)
    init_names = {i.name for i in shaped.graph.initializer}

    print('\n' + '=' * 65)
    print('  SHAPE ANALYSIS REPORT')
    print('=' * 65)

    # Count symbolic vs fixed dims
    n_symbolic = 0
    n_fixed = 0
    n_unknown = 0
    sym_names = set()

    def count_dims(type_proto):
        nonlocal n_symbolic, n_fixed, n_unknown
        t = type_proto.tensor_type
        if t.HasField('shape'):
            for d in t.shape.dim:
                if d.dim_param:
                    n_symbolic += 1
                    sym_names.add(d.dim_param)
                elif d.dim_value > 0:
                    n_fixed += 1
                else:
                    n_unknown += 1

    for inp in shaped.graph.input:
        if inp.name not in init_names:
            count_dims(inp.type)
    for vi in shaped.graph.value_info:
        count_dims(vi.type)
    for out in shaped.graph.output:
        count_dims(out.type)

    print(f'\n  Dimension Statistics:')
    print(f'    Fixed:    {n_fixed}')
    print(f'    Symbolic: {n_symbolic} ({sym_names})')
    print(f'    Unknown:  {n_unknown}')

    # Parameter count
    total_params = sum(to_array(i).size for i in shaped.graph.initializer)
    total_bytes = sum(to_array(i).nbytes for i in shaped.graph.initializer)
    print(f'\n  Parameters:')
    print(f'    Count: {total_params:,}')
    print(f'    Size:  {total_bytes:,} bytes ({total_bytes/1024:.1f} KB)')

    # Data flow table
    print(f'\n  Data Flow:')
    print(f'  {"Tensor":15s} {"Kind":12s} {"Type":8s} {"Shape"}')
    print(f'  {"-"*55}')
    for inp in shaped.graph.input:
        if inp.name in init_names:
            continue
        t = inp.type.tensor_type
        dtype = TensorProto.DataType.Name(t.elem_type)
        dims = [d.dim_param or d.dim_value for d in t.shape.dim] if t.HasField('shape') else '?'
        print(f'  {inp.name:15s} {"input":12s} {dtype:8s} {dims}')
    for vi in shaped.graph.value_info:
        t = vi.type.tensor_type
        dtype = TensorProto.DataType.Name(t.elem_type)
        dims = [d.dim_param or d.dim_value for d in t.shape.dim] if t.HasField('shape') else '?'
        print(f'  {vi.name:15s} {"intermediate":12s} {dtype:8s} {dims}')
    for out in shaped.graph.output:
        t = out.type.tensor_type
        dtype = TensorProto.DataType.Name(t.elem_type)
        dims = [d.dim_param or d.dim_value for d in t.shape.dim] if t.HasField('shape') else '?'
        print(f'  {out.name:15s} {"output":12s} {dtype:8s} {dims}')

    # Runtime test
    if run_test and sym_names:
        print(f'\n  Dynamic Shape Test:')
        sess = ort.InferenceSession(
            model.SerializeToString(), providers=['CPUExecutionProvider'])
        runtime_inputs = [i for i in model.graph.input if i.name not in init_names]
        for trial in [1, 8, 32]:
            feeds = {}
            for inp in runtime_inputs:
                shape = []
                for d in inp.type.tensor_type.shape.dim:
                    if d.dim_param:
                        shape.append(trial)
                    else:
                        shape.append(d.dim_value)
                dt = tensor_dtype_to_np_dtype(inp.type.tensor_type.elem_type)
                feeds[inp.name] = np.random.randn(*shape).astype(dt)
            result = sess.run(None, feeds)[0]
            print(f'    sym={trial}: output shape={result.shape}')

    print('\n' + '=' * 65)

from onnx.helper import tensor_dtype_to_np_dtype
shape_analyzer(model_post)

---

## Summary

| Exercise | Topic | Key Takeaway |
|----------|-------|-------------|
| 1 | Symbolic dims | `'N'` enables dynamic batching |
| 2 | Shape inference | `infer_shapes()` fills intermediates |
| 3 | Type inspection | Walk TypeProto for every tensor |
| 4 | Mixed types | FLOAT data + INT64 indices + BOOL masks |
| 5 | Concrete vs symbolic | Flexibility vs specificity |
| 6 | Shape debugging | Detect dimension mismatches |
| 7 | Dynamic axis test | Latency scales with input size |
| Challenge | Shape analyzer | Complete shape audit tool |

**Next:** [ONNX Model Format and Protobuf →](../../04_ONNX_Model_Format_and_Protobuf/)